In [ ]:
%reload_ext autoreload
%autoreload 2

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

import os
import logging
import yaml
import requests
import sqlite3
import ipynbname

import pandas as pd

import ferdinand.utils as utl   
import ferdinand.sqlite_utils as sqlutl
import ferdinand.download_utils as dlutl

from collections import defaultdict
from pathlib import Path
from types import SimpleNamespace

# Purpose and Documentation


## Purpose

This notebook 
- fetches metadata rows for image parameter(s) defined in _../config/config.yml_
- iterates over IMPC phenotyping centers and saves center-specific raw CSV files to _../data/&lt;run_name&gt;/&lt;center_data_dir&gt;_
- performs QC and merges the results into a single consolidated CSV file stored at _../data/&lt;run_name&gt;/&lt;all_data_dir&gt;/&lt;sqlite_db&gt;.csv_, which is then loaded into the SQLite database _&lt;sqlite_db&gt;.db_.

## Location

_&lt;project_dir&gt;/notebooks/010_fetch_data_from_impc.ipynb_

## Useful links

| # | URL  | description |
| - | :--- | :---        |
| 1 |https://www.mousephenotype.org/impress/index                                        | IMPReSS               |
| 2 |https://www.mousephenotype.org/help/programmatic-data-access/access-patterns/       | API access patterns   |
| 3 |https://www.mousephenotype.org/help/programmatic-data-access/raw-experimental-data/ | how to get raw data   |
|   |https://www.mousephenotype.org/help/programmatic-data-access/images/                | image API             |
| 4 |https://www.mousephenotype.org/help/programmatic-data-access/data-fields/           | explanation of fields |
| 5 | ---                                                                                | ---                   |
| 6 |https://api.mousephenotype.org/impress/                                             | list of APIs          |
| 7 |https://www.mousephenotype.org/phenodcc/impress-api-documentation/                  | API documentation     |
| 8 |https://www.ebi.ac.uk/mi/impc/solr/experiment/select?q=*%3A*&rows=0&facet=on&facet.field=phenotyping_center| get distinct phenotyping centers|
| 9 | --- | --- |
| 10| https://api.mousephenotype.org/impress/pipeline/list/full| all pipelines |


## Prerequisites

- A valid runtime configuration file (_../config/config.yml_) with correctly defined _centers_, _run_name_, _parameters_, and _directories_
- Successful execution of the template setup cells from _NNN_notebook_template.ipynb_
- Existing _../data_, and _../reports_ directories 
- Network access to the IMPC Solr API (<a href="https://www.ebi.ac.uk/mi/impc/solr/impc_images/select" target="_blank">https://www.ebi.ac.uk/mi/impc/solr/impc_images/select</a>)

</p>

# Definition / Settings

This section initializes the notebook environment: 
- loads the runtime configuration from _../config/config.yml_
- creates the required directory structure
- sets up logging and the SQLite connection

**Do not proceed past this section** </br>
Missing paths and an uninitialized database connection will cause subsequent cells to crash.

## General settings

In [ ]:
# load the configuration file
config_path = Path('../config/config.yml')
with open(config_path, 'r') as file:
    config = yaml.safe_load(file)

print(pd.DataFrame(config['settings'].items(), columns=['configuration parameter', 'set to']).to_markdown(tablefmt="pretty", index=False))

## Set up runtime paths

In [ ]:
# create directories as per configuration
dirs = SimpleNamespace(**config['data_directories'])
for key, value in vars(dirs).items():
    dirs.__dict__[key] = os.path.join(config['project_root'], config['data_dir'], config['run_name'], value)
    os.makedirs(dirs.__dict__[key], exist_ok=True)
    if os.path.exists(dirs.__dict__[key]):
        print(f" Directory created: {dirs.__dict__[key]}")
    else:
        print(f" ERROR: Directory does not exist: {dirs.__dict__[key]}")

## Store runtime configuration when necessary

In [ ]:
# copy the current '../config/config.yml' to '../reports/'
dest_filename = f"config_{config['run_name']}.yml"
dest_dir = os.path.join(config['project_root'], config['reports_dir'], config['run_name'])
os.makedirs(dest_dir, exist_ok=True)
dest_path = os.path.join(dest_dir, dest_filename)
if not os.path.exists(dest_path): 
    commmand_string = f"cp {config_path} {dest_path}"
    os.system(commmand_string)

    if os.path.exists(dest_path): 
        print(f" Current runtime configuration \'{config_path}' saved to \'{dest_path}\'")
    else:
        print(f" ERROR: could not save configuration file to \'{dest_path}\'")

## Set up logger

In [ ]:
# setup logging
notebook_name = ipynbname.name()

log_path = os.path.join(config['project_root'], config['reports_dir'], config['run_name'], 'logs')
os.makedirs(log_path, exist_ok=True)

log_file = os.path.join(log_path, f"{notebook_name}.log")
print(f"Log file: {log_file}")

try:
    logger = utl.setup_logger(log_file, log_level=logging.INFO)
    print("Logger successfully set up.")
except Exception as e:
    print(f"Error setting up logger: {e}")

## Set up SQLite connection

In [ ]:
# expected SQLlite database file
sqlite_file = os.path.join(dirs.all_data_dir, f"{config['sqlite_db']}.db")

# if already exists, connect to it
if os.path.exists(sqlite_file):
    conn = sqlutl.connect_db(sqlite_file)
    if conn is not None:
        print(f" Connected to database {sqlite_file}\n")
else: 
    print(f" ERROR: No SQLite database found!")

## Download settings

In [ ]:
# this is the base URL for the IMPC raw data API
api_base_url = "https://www.ebi.ac.uk/mi/impc/solr/impc_images/select?q=parameter_stable_id:"

# example: get one row of data
# https://www.ebi.ac.uk/mi/impc/solr/impc_images/select?q=parameter_stable_id:IMPC_XRY_034_001%20AND%20colony_id:MEAW&wt=csv&rows=1

# here we define which filters we want to apply
api_filters = ("")

# here we define which fields we want to have
api_fields  = ("&fl=phenotyping_center,pipeline_name,pipeline_stable_id,life_stage_name,external_sample_id,date_of_birth,"
               "gene_symbol,sex,biological_sample_group,strain_name,procedure_name,procedure_stable_id,age_in_days,"
               "age_in_weeks,parameter_stable_id,parameter_name,litter_id,gene_accession_id,zygosity,file_type,"
               "omero_id,date_of_experiment,parameter_association_name,parameter_association_stable_id,parameter_association_value")

if 'additional_api_fields' in config.keys() and config['additional_api_fields'] is not None:
    api_fields += ',' + ','.join(config['additional_api_fields'])

# here we define additional options
api_options  = '&start=0&wt=csv'

# mode: REAL=download everything, TEST=only 20 rows
mode = config["run_mode"]

# Fetch metadata from IMPC

In [ ]:
# load parameters from config file
my_parameters = pd.DataFrame(config['parameters'])
my_parameters

In [ ]:
# load centers from config file
my_centers = config['centers']
my_centers

## Expected parameter values by center

In [ ]:
# initialise dictionary
centers_numbers = defaultdict(lambda: defaultdict(int))

total_number = 0

# loop over all parameters
for my_stable_id in my_parameters.stable_id:
    
    total_number = 0
    
    my_url = f"https://www.ebi.ac.uk/mi/impc/solr/impc_images/select?q=parameter_stable_id:{my_stable_id}&rows=0&facet=on&facet.field=phenotyping_center"
    
    response = requests.get(my_url)

    if response.status_code == 200:
        data = response.json()

        numbers = data["facet_counts"]["facet_fields"]["phenotyping_center"]

        # build dictionary centers_numbers[center][parameter]
        for i in range(0, len(numbers), 2):
            centers_numbers[numbers[i]][my_stable_id] = numbers[i+1]
            total_number += numbers[i+1]

# convert dictionary to dataframe
expected_df = pd.DataFrame(centers_numbers)
print(f"total number of images: {total_number}")

# show the dataframe
expected_df

## Data per center and parameter

In [ ]:
# now we have the expected number of images per center and parameter, we can loop over all parameters and centers and download the metadata
dlutl.fetch_metadata(mode=mode, 
                     my_centers=my_centers, 
                     my_parameters=my_parameters, 
                     expected_df=expected_df, 
                     dirs=dirs, 
                     api_base_url=api_base_url, 
                     api_fields=api_fields, 
                     api_options=api_options, 
                     logger=logger)

# Iterate centers, run QC and combine outputs into one file

In [ ]:
# now we have downloaded the metadata for all centers and parameters, we can merge the metadata files into one dataframe
all_data_df = dlutl.merge_metadata_files(config, dirs, logger)

In [ ]:
print(f" {all_data_df.mouse_id.nunique()} \tdistinct mouse IDs")
print(f" {all_data_df.omero_id.nunique()}\tdistinct OMERO IDs")
print(f" {all_data_df['gene_symbol'].nunique()} \tdistinct genes")

## Write DataFrame to SQLite database

In [ ]:
# show the first 5 rows of the all data dataframe
all_data_df.head(5)

In [ ]:
# now we can save the all data dataframe to a SQLlite database
conn = sqlite3.connect(sqlite_file)
all_data_df.to_sql("impc_data", conn, if_exists="replace", index=False)
conn.close()